In [0]:
from pyspark.sql import functions as F

In [0]:
film_info = spark.read.table("films.bronze.film")

In [0]:
### on récupère si le film appartient à une collection

film_info = film_info.withColumn("collection_name", F.coalesce(F.col("belongs_to_collection.name"),F.lit("Unique"))).drop("belongs_to_collection")

In [0]:
### nouvelle table film_genre : on explode les genres

film_genre = film_info.select("*",F.explode("genres").alias("genre"))
film_genre = film_genre.select("*",
                             F.col("genre.id").alias("genre_id"),
                             F.col("genre.name").alias("genre_name")
                             ).drop("genre","genres")

In [0]:
### nouvelle table film_country : on explode les pays d'origines

film_country = film_info.select("*", F.explode("origin_country").alias("origin_country_name")).drop("origin_country")

In [0]:
## On garde que les colonnes voulues pour chaque table

film_info = film_info.select("id","budget","original_language","popularity","release_date","revenue","runtime","title","vote_average","vote_count","collection_name")
film_genre = film_genre.select("id","genre_id","genre_name")
film_country = film_country.select("id","origin_country_name")

In [0]:
## on load tout ça dans silver
film_info.write.mode("overwrite").saveAsTable("films.silver.film_info")
film_genre.write.mode("overwrite").saveAsTable("films.silver.film_genre")
film_country.write.mode("overwrite").saveAsTable("films.silver.film_country")